In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database("bootcamp")
session.use_schema("atp524")  # Replace with your schema.

In [ ]:
import importlib.util
import sys
from pathlib import Path

# The scripts have numbered filenames (e.g. 03.ingest_sec_ground_truth.py), which
# `import` cannot load, so load them by path under their original module names.
cwd = Path.cwd().resolve()
python_dir = next(
    d for d in (cwd, cwd / "python", *cwd.parents)
    if (d / "03.ingest_sec_ground_truth.py").exists()
)
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))  # for their own imports, e.g. sec_facts

def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, python_dir / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

ingest_sec_ground_truth = load_module("ingest_sec_ground_truth", "03.ingest_sec_ground_truth.py")
ingest_annual_report_pdfs = load_module("ingest_annual_report_pdfs", "02.ingest_annual_report_pdfs.py")

# SEC contact string: the SEC asks automated requests to identify a contact.
# Each ingestion file already sets SEC_USER_AGENT near its top; setting it here only
# overrides it for ingestion run from this notebook session (e.g. via 00.run_pipeline).
# Use your own address, e.g. "AnnualReportsLab your.real.contact@example.com".
ingest_sec_ground_truth.SEC_USER_AGENT = "FinancialDocumentsCourse zach@dataexpert.io"
ingest_annual_report_pdfs.SEC_USER_AGENT = "FinancialDocumentsCourse zach@dataexpert.io"

In [ ]:
-- Choose what to process: the ONLY place the ticker, year range and metric set are set.
-- 02 (PDFs) and 03 (SEC facts) ingest this scope, and every SQL step from 04 on
-- processes every report in it, so no ingested year is silently skipped.
-- Check coverage with pipeline_coverage (06.check_readiness.sql).
-- To process a single report, set first_year = last_year (a range covers at most six years).
-- include_all_metrics: FALSE = the starter metrics (assets, revenue, diluted EPS;
--   metric_catalog.is_starter), TRUE = all 10 catalog metrics. Readiness (06), extraction
--   (08) and report_status (09) all read it through the scope_metrics view, so they always
--   agree. With TRUE, a report is READY only if SEC has a value for all 10 metrics.
-- ATP NOTE: include_all_metrics - resting value FALSE. Set TRUE only for a 10-metric run
--   (08 then re-extracts, because the questions change), and set it back afterwards.
-- lab_settings holds exactly one row (setting_id = 1); this MERGE only ever updates that row.
USE WAREHOUSE compute_wh;
MERGE INTO bootcamp.atp524.lab_settings t
USING (SELECT 1 AS setting_id, 'AAPL' AS ticker, 2021 AS first_year, 2025 AS last_year,
              FALSE AS include_all_metrics) s
   ON t.setting_id = s.setting_id
WHEN MATCHED THEN UPDATE SET ticker = s.ticker, first_year = s.first_year,
                             last_year = s.last_year, include_all_metrics = s.include_all_metrics,
                             updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (setting_id, ticker, first_year, last_year, include_all_metrics)
                      VALUES (s.setting_id, s.ticker, s.first_year, s.last_year, s.include_all_metrics);

SELECT setting_id, ticker, first_year, last_year, include_all_metrics, updated_at
FROM bootcamp.atp524.lab_settings;